<a href="https://colab.research.google.com/github/aravinth-xuno/fraud-detection-notebook/blob/main/Pipeline_Data_Transformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Data Loading and Environment Setup
In this section, we import the necessary libraries, mount Google Drive to access our datasets, and load the raw transaction data into a pandas DataFrame.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
train_df = pd.read_csv("/content/drive/Shareddrives/transaction-training-set/dataset/txn_prod_2025-12-24.csv", low_memory=False)

In [ ]:
train_df.shape

(174466, 29)

## 2. Data Preprocessing
To perform time-based calculations, we convert the `INITIATED_AT` column to datetime objects and sort the dataset by `USER_ID` and time. This ensures that our rolling window operations are logically consistent within each user's history.

In [ ]:
train_df['INITIATED_AT'] = pd.to_datetime(train_df['INITIATED_AT'], format="ISO8601", utc=True)

In [ ]:
train_df = train_df.sort_values(by=['USER_ID', 'INITIATED_AT']).reset_index(drop=True)

In [ ]:
grouped = train_df.groupby('USER_ID')

## 3. Feature Engineering:

### 3.1 High Velocity Transactions
We calculate several metrics to capture user activity velocity over different time horizons:

* **last_1hour_count**: Number of transactions in the previous 60 minutes.
* **last_24hour_count**: Total transactions within the last 24 hours.
* **last_30days_count**: Total transactions over the past 30 days.
* **ratio_1hr_24hr**: The intensity of recent activity (1h) compared to the 24h average.
* **ratio_24hr_30day**: The intensity of daily activity compared to the 30-day average.

These features help identify sudden spikes in transaction frequency which are key indicators of high-velocity users.

In [ ]:
train_df['cnt_1h'] = grouped.rolling('1h', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

In [ ]:
train_df['cnt_total_25h'] = grouped.rolling('25h', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

In [ ]:
train_df['last_24hour_count'] = grouped.rolling('24h', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

In [ ]:
train_df['last_30days_count'] = grouped.rolling('30D', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

In [ ]:
train_df['cnt_total_31d'] = grouped.rolling('31D', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

In [ ]:
train_df['last_1hour_count'] = train_df['cnt_1h']

In [ ]:
train_df['last_24hour_count_shifted'] = train_df['cnt_total_25h'] - train_df['cnt_1h']

In [ ]:
train_df['last_30days_count_shifted'] = train_df['cnt_total_31d'] - train_df['last_24hour_count']

In [ ]:
train_df.drop(columns=['cnt_total_25h', 'cnt_total_31d'], inplace=True)

### 3.2 Dormancy and Transaction Behavior Features
These features focus on identifying patterns of inactivity or deviations in spending behavior:

* **DSL_TX (Days Since Last Transaction)**: Calculates the time elapsed between consecutive transactions for each user. This helps identify how long a user has been inactive.
* **DSL_LGN (Days Since Last Login)**: An estimated metric assuming a login occurs approximately 3 minutes before a transaction. It represents the time since the user's likely last login.
* **txn_amount_usd**: Converts the `SENDING_AMOUNT` to a standard float format for numerical analysis.
* **txn_amount_zscore**: Measures how many standard deviations a transaction amount is from the user's personal mean. This is crucial for detecting outliers or sudden changes in spending volume for a specific account.

In [ ]:
train_df['DSL_TX'] = (
    train_df.groupby('USER_ID')['INITIATED_AT']
    .diff()
    .dt.total_seconds()
    .div(86400)
    .fillna(0)
    .astype('float32')
)

In [ ]:
login_df = pd.read_csv(
    '/content/drive/Shareddrives/transaction-training-set/dataset/login_prod_2026-02-27.csv',
    low_memory = False
  )

login_df['LOGIN_DT'] = pd.to_datetime(login_df['LOGIN_DT'], format="ISO8601", utc=True)

login_df = login_df.sort_values(['USER_ID', 'LOGIN_DT'])
login_df['USER_ID'] = login_df['USER_ID'].astype(str)

login_df = login_df[login_df['LOGIN_SUCCESSFUL'] == True]
dsl_lgn_values = []

for user_id, user_txns in train_df.groupby('USER_ID', sort=False):
    user_logins = login_df[login_df['USER_ID'] == user_id]['LOGIN_DT']

    for txn_time in user_txns['INITIATED_AT']:
        past_logins = user_logins[user_logins < txn_time]

        if len(past_logins) >= 2:
            last_login_time = past_logins.iloc[-1]       # most recent login before txn
            previous_login_time = past_logins.iloc[-2]   # login before that

            # Compute difference in days
            dsl_lgn = (last_login_time - previous_login_time).total_seconds() / 86400.0
        else:
            dsl_lgn = 0.0

        # Append to results
        dsl_lgn_values.append(dsl_lgn)

train_df['DSL_LGN'] = pd.Series(dsl_lgn_values, index=train_df.index, dtype='float32')


In [ ]:
train_df['txn_amount_usd'] = train_df['SENDING_AMOUNT'].astype(float)

In [ ]:
def z_score(x):
    return (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) != 0 else 0

In [ ]:
train_df['txn_amount_zscore'] = train_df.groupby('USER_ID')['txn_amount_usd'].transform(z_score)

## 4. Ratio and Log-Transform Features
We derive relative activity metrics by calculating ratios between different time windows (e.g., 1h vs 24h activity). Log transformations are applied to normalize these ratios, which can help model performance.

In [ ]:
train_df['ratio_1hr_24hr'] = (train_df['last_1hour_count'] + 1) / (train_df['last_24hour_count_shifted'] + 1)

In [ ]:
train_df['ratio_24hr_30day'] = (train_df['last_24hour_count'] + 1) / (train_df['last_30days_count_shifted'] + 1)

In [ ]:
train_df['log_ratio_1hr_24hr'] = np.log1p(train_df['ratio_1hr_24hr'])

In [ ]:
train_df['log_ratio_24hr_30day'] = np.log1p(train_df['ratio_24hr_30day'])

### 5. Final Feature Selection and Export
Finally, we review the statistical summary of our newly created features and export the augmented dataset back to Google Drive for model training.

In [ ]:
transformed_columns = [
    "last_1hour_count",
    "last_24hour_count",
    "last_30days_count",
    "last_24hour_count_shifted",
    "last_30days_count_shifted",
    "ratio_1hr_24hr",
    "log_ratio_1hr_24hr",
    "ratio_24hr_30day",
    "log_ratio_24hr_30day",
    "DSL_TX",
    "DSL_LGN"
    "txn_amount_usd",
    "txn_amount_zscore"
]

In [ ]:
features = [
    "last_1hour_count",
    "last_24hour_count",
    "last_30days_count",
    "ratio_1hr_24hr",
    "ratio_24hr_30day",
    "DSL_TX",
    "DSL_LGN",
    "txn_amount_usd",
    "txn_amount_zscore"
]

In [ ]:
train_df[features].head()

,last_1hour_count,last_24hour_count,last_30days_count,ratio_1hr_24hr,ratio_24hr_30day,DSL_TX,DSL_LGN,txn_amount_usd,txn_amount_zscore
0,0.0,0.0,0.0,1.0,1.0,0.000000,0.0,1001.0,-1.058548
1,0.0,0.0,0.0,1.0,1.0,94.928795,0.0,4000.0,0.264527
2,0.0,0.0,1.0,1.0,0.5,20.978170,0.0,5500.0,0.926285
3,0.0,0.0,0.0,1.0,1.0,113.304535,0.0,6000.0,1.146871
4,0.0,0.0,0.0,1.0,1.0,294.037872,0.0,501.0,-1.279134


In [ ]:
train_df[features].describe()

,last_1hour_count,last_24hour_count,last_30days_count,ratio_1hr_24hr,ratio_24hr_30day,DSL_TX,DSL_LGN,txn_amount_usd,txn_amount_zscore
count,174466.000000,174466.000000,174466.000000,174466.000000,174466.000000,174466.000000,174466.000000,174466.000000,1.744660e+05
mean,0.082228,0.112016,0.991248,1.066444,0.794521,61.349583,1.060459,1296.520044,5.152490e-16
std,0.323289,0.406943,2.420610,0.329968,0.440341,141.428040,7.852910,1827.413962,9.907201e-01
min,0.000000,0.000000,0.000000,0.066667,0.011236,0.000000,0.000000,1.050000,-7.850997e+00
25%,0.000000,0.000000,0.000000,1.000000,0.500000,6.396337,0.000000,250.000000,-5.846248e-01
50%,0.000000,0.000000,1.000000,1.000000,1.000000,25.433298,0.000000,551.000000,-2.996719e-01
75%,0.000000,0.000000,1.000000,1.000000,1.000000,57.529913,0.000000,1500.000000,3.072525e-01
max,9.000000,14.000000,88.000000,10.000000,10.000000,3333.321533,235.824692,10000.000000,1.086312e+01


In [ ]:
train_df.to_csv("/content/drive/Shareddrives/transaction-training-set/temp_working_directory/transformed_dataset.csv", index=False)